# Crypto Backtest Module - Example Usage

This notebook demonstrates how to use the crypto_backtest module for backtesting trading strategies.

In [ ]:
# Import the module
import sys
sys.path.append('..')

from crypto_backtest import (
    BinanceLoader,
    CsvLoader,
    SmaCrossover,
    RsiStrategy,
    BollingerStrategy,
    BacktestEngine,
    BacktestDashboard,
    GridOptimizer,
    PerformanceMetrics
)

import pandas as pd
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data from Binance

In [ ]:
# Initialize Binance loader
loader = BinanceLoader()

# Load BTC data
df = loader.load(
    symbol="BTCUSDT",
    timeframe="1h",
    start="2024-01-01",
    end="2024-12-31"
)

print(f"Loaded {len(df)} candles")
df.head()

## 2. Simple Moving Average Crossover Strategy

In [ ]:
# Create strategy
strategy = SmaCrossover(fast_period=10, slow_period=30)

# Create backtest engine
engine = BacktestEngine(
    strategy=strategy,
    data=df,
    initial_capital=10000,
    commission=0.001,
    symbol="BTCUSDT",
    timeframe="1h"
)

# Run backtest in fast mode (quick metrics)
result_fast = engine.run(mode="fast")

print("\nPerformance Metrics:")
print(PerformanceMetrics.summary_table(result_fast.metrics))

## 3. Full Backtest with Visualization

In [ ]:
# Run in full mode for detailed visualization
result_full = engine.run(mode="full")

# Display dashboard
dashboard = BacktestDashboard(result_full)
dashboard.show()

## 4. View Individual Trades

In [ ]:
# Show trades table
trades_df = dashboard.show_trades_table()
print(f"\nTotal Trades: {len(trades_df)}")
trades_df.head(10)

## 5. RSI Strategy Example

In [ ]:
# Create RSI strategy
rsi_strategy = RsiStrategy(rsi_period=14, oversold=30, overbought=70)

# Run backtest
rsi_engine = BacktestEngine(
    strategy=rsi_strategy,
    data=df,
    initial_capital=10000,
    symbol="BTCUSDT",
    timeframe="1h"
)

rsi_result = rsi_engine.run(mode="full")

# Display results
print("\nRSI Strategy Results:")
print(PerformanceMetrics.summary_table(rsi_result.metrics))

## 6. Parameter Optimization

In [ ]:
# Optimize SMA crossover parameters
optimizer = GridOptimizer(
    strategy_class=SmaCrossover,
    data=df,
    param_grid={
        "fast_period": [5, 10, 15, 20],
        "slow_period": [30, 40, 50, 60]
    },
    metric="sharpe_ratio",
    initial_capital=10000
)

# Run optimization
results = optimizer.run()

# Show summary
optimizer.summary(top_n=5)

## 7. Compare Multiple Strategies

In [ ]:
# Run multiple strategies
strategies = [
    ("SMA 10/30", SmaCrossover(fast_period=10, slow_period=30)),
    ("SMA 20/50", SmaCrossover(fast_period=20, slow_period=50)),
    ("RSI 14", RsiStrategy(rsi_period=14)),
    ("Bollinger 20", BollingerStrategy(period=20))
]

results_list = []
for name, strat in strategies:
    eng = BacktestEngine(strat, df, initial_capital=10000)
    res = eng.run(mode="fast")
    res.strategy_name = name
    results_list.append(res)

# Compare results
comparison = PerformanceMetrics.comparison_table(results_list)
print("\nStrategy Comparison:")
print(comparison)

## 8. Custom Strategy Example

In [ ]:
from crypto_backtest import BaseStrategy

class CustomMomentumStrategy(BaseStrategy):
    """Simple momentum strategy based on price change."""
    
    default_params = {"lookback": 20, "threshold": 0.02}
    
    def indicators(self, df):
        df = df.copy()
        # Calculate momentum
        df["momentum"] = df["close"].pct_change(self.params.lookback)
        return df
    
    def signal(self, df):
        df = self.indicators(df)
        signals = pd.Series(0, index=df.index)
        
        # Buy on strong positive momentum
        signals[df["momentum"] > self.params.threshold] = 1
        
        # Sell on strong negative momentum
        signals[df["momentum"] < -self.params.threshold] = -1
        
        return signals

# Test custom strategy
custom_strategy = CustomMomentumStrategy(lookback=20, threshold=0.02)
custom_engine = BacktestEngine(custom_strategy, df, initial_capital=10000)
custom_result = custom_engine.run(mode="full")

print("\nCustom Strategy Results:")
print(PerformanceMetrics.summary_table(custom_result.metrics))

## 9. Load Data from CSV

In [ ]:
# Example: Load from CSV file
# csv_loader = CsvLoader(
#     filepath="path/to/your/data.csv",
#     column_map={
#         "timestamp": "date",
#         "open": "open",
#         "high": "high",
#         "low": "low",
#         "close": "close",
#         "volume": "volume"
#     }
# )
# csv_df = csv_loader.load(start="2024-01-01", end="2024-12-31")
# print(csv_df.head())

## 10. Save Results

In [ ]:
# Save dashboard as HTML
dashboard.save_html("backtest_dashboard.html")
print("Dashboard saved!")

# Save metrics to CSV
metrics_df = PerformanceMetrics.summary_table(result_full.metrics)
metrics_df.to_csv("backtest_metrics.csv", index=False)
print("Metrics saved!")

# Save trades to CSV
trades_df = dashboard.show_trades_table()
trades_df.to_csv("backtest_trades.csv", index=False)
print("Trades saved!")